In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import random
from typing import Dict, List

[2025-08-31 08:16:15,711] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/data/yuhui/miniconda3/envs/cot/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/data/yuhui/miniconda3/envs/cot/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


[2025-08-31 08:16:16,746] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [3]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# MODEL_NAME = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype =torch.float16,
# )

In [4]:
DATASET_SIZE = 2000
MAX_LENGTH = 2048  # Maximum length for the model input
size = DATASET_SIZE if DATASET_SIZE > 0 else 1000
dataset = load_dataset("cais/mmlu", "all", split="test")
dataset = dataset.select(range(size)) if size < len(dataset) else dataset

index = 507
example = dataset[index]
question = example["question"]
choices = example["choices"]
correct_answer_idx = example["answer"]
wrong_answer_idx = (correct_answer_idx + 1) % len(choices)  # Give the wrong answer index

# Format choices as A, B, C, D
choice_labels = ["A", "B", "C", "D", "E", "F", "G", "H"][:len(choices)]
formatted_choices = []
for i, choice in enumerate(choices):
    formatted_choices.append(f"({choice_labels[i]}) {choice}")

choices_text = "\n".join(formatted_choices)
correct_answer = choice_labels[correct_answer_idx]
wrong_answer = choice_labels[wrong_answer_idx]

# Create conversation format for input (without assistant response)
input_messages = [
    {
        "role": "user", 
        "content": f"What is the correct answer to this question? Question:\n {question}\nChoices:\n{choices_text}\nOutput format: The correct answer is (A/B/C/D)."
    }
]

correct_text = f"The correct answer is ({correct_answer}) {choices[correct_answer_idx]}"
wrong_text = f"The correct answer is ({wrong_answer}) {choices[wrong_answer_idx]}"

input_text = tokenizer.apply_chat_template(
    input_messages,
    tokenize=False,
    add_generation_prompt=True  # This adds the assistant prompt
)


# full_correct_text = input_text + "<think>\n\n<think>\n\n" + correct_text + "<|end|>"

# input_text + "<|im_start|>assistant\n<think>\n\n</think>\n\n" + correct_text + "<|im_end|>\n"
full_wrong_text = input_text + "<think>\n\n</think>\n\n" + wrong_text + "<|end|>"
# full_tokenized_correct = tokenizer(
#     full_correct_text,
#     truncation=True,
#     padding=False,
#     max_length=MAX_LENGTH,
#     return_tensors=None,
#     add_special_tokens=False,  # Don't add special tokens to avoid duplication
# )

full_tokenized_wrong = tokenizer(
    full_wrong_text,
    truncation=True,
    padding=False,
    max_length=MAX_LENGTH,
    return_tensors=None,
    add_special_tokens=False,  # Don't add special tokens to avoid duplication
)

input_tokenized = tokenizer(
    input_text,
    truncation=True,
    padding=False,
    max_length=MAX_LENGTH,
    return_tensors=None,
    add_special_tokens=False,  # Don't add special tokens to avoid duplication
)

In [5]:
# print(f"full_tokenized_correct: {full_tokenized_correct}")
print(f"full_tokenized_wrong: {full_tokenized_wrong}")
print(f"input_tokenized: {input_tokenized}")

full_tokenized_wrong: {'input_ids': [151646, 151644, 3838, 374, 279, 4396, 4226, 311, 419, 3405, 30, 15846, 510, 3555, 374, 264, 1375, 642, 58804, 5267, 89283, 510, 4346, 8, 8536, 58804, 624, 5349, 8, 3984, 15439, 58804, 624, 3025, 8, 62861, 58804, 624, 5432, 8, 2869, 531, 552, 315, 279, 10578, 323, 8557, 3376, 518, 279, 32171, 624, 5097, 3561, 25, 576, 4396, 4226, 374, 320, 32, 16276, 11295, 14953, 568, 151645, 151648, 198, 151648, 271, 151649, 271, 785, 4396, 4226, 374, 320, 32, 8, 8536, 58804, 15757, 91, 408, 91, 29], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
input_tokenized: {'input_ids': [151646, 151644, 3838, 374, 279, 4396, 4226, 311, 419, 3405, 30, 15846, 510, 3555, 374, 264, 1375, 642, 58804, 5267, 89283, 510, 4346, 8, 8536, 58804, 624, 5349,

In [6]:
tokenizer.decode(full_tokenized_wrong["input_ids"][len(input_tokenized["input_ids"]) + 9:])  # Decode to check the content + 10

'A) Hand fracture.<|end|>'

In [7]:
tokenizer.decode(input_tokenized["input_ids"])

'<｜begin▁of▁sentence｜><｜User｜>What is the correct answer to this question? Question:\n What is a colles fracture?\nChoices:\n(A) Hand fracture.\n(B) Elbow fracture.\n(C) Finger fracture.\n(D) Fracture of the radius and ulna at the wrist.\nOutput format: The correct answer is (A/B/C/D).<｜Assistant｜><think>\n'

In [6]:
tokenizer.encode("\n</think>\n\n" + f"The correct answer is")

[151646, 198, 151649, 271, 785, 4396, 4226, 374]

In [9]:
tokenizer.decode([350, 32, 8, 1843, 290, 67428, 6251, 328, 290, 11089, 25709, 13])

' (A) If the incidence rate of the disease falls.'

In [9]:
len([32, 8, 34303, 678, 525, 5147, 80524, 20336, 13])

9

In [7]:
tokenizer.decode(input_tokenized["input_ids"])  # Decode to check the content

'<|system|>Your name is Phi, an AI math expert developed by Microsoft.<|end|><|user|>What is the correct answer to this question? Question:\n What is a colles fracture?\nChoices:\n(A) Hand fracture.\n(B) Elbow fracture.\n(C) Finger fracture.\n(D) Fracture of the radius and ulna at the wrist.\nOutput format: The correct answer is (A/B/C/D).<|end|><|assistant|>'

In [9]:
correct_text = f"The correct answer is ({correct_answer}) {choices[correct_answer_idx]}"
wrong_text = f"The correct answer is ({wrong_answer}) {choices[wrong_answer_idx]}"
print(f"Correct answer text: {correct_text}")
print(f"Wrong answer text: {wrong_text}")

Correct answer text: The correct answer is (D) Fracture of the radius and ulna at the wrist.
Wrong answer text: The correct answer is (A) Hand fracture.


In [10]:
full_correct_text = input_text + "<｜Assistant｜><think>\n\n<think>\n\n" + correct_text + "<｜end▁of▁sentence｜>"
full_correct_text

'<｜begin▁of▁sentence｜><｜User｜>What is the correct answer to this question? Question:\n What is a colles fracture?\nChoices:\n(A) Hand fracture.\n(B) Elbow fracture.\n(C) Finger fracture.\n(D) Fracture of the radius and ulna at the wrist.<｜Assistant｜><think>\n\n<think>\n\nThe correct answer is (D) Fracture of the radius and ulna at the wrist.<｜end▁of▁sentence｜>'

In [11]:
tokenizer.decode([151644, 872, 198, 3838, 374, 279, 4396, 4226, 311, 419, 3405, 30, 15846, 510, 758, 264, 1990, 62105, 41930, 315, 15552, 11, 279, 11341, 315, 3999, 1543, 549, 4510, 6283, 307, 1543, 374, 510, 89283, 510, 4346, 8, 3890, 624, 5349, 8, 10838, 553, 279, 2331, 8500, 304, 40114, 624, 3025, 8, 50933, 10838, 624, 5432, 8, 2677, 220, 16, 25, 16, 13, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 785, 4396, 4226, 374, 320, 32, 8, 3890, 13, 151645, 198])

'<｜User｜>user\nWhat is the correct answer to this question? Question:\n In a double stranded molecule of DNA, the ratio of purines : pyrimidines is:\nChoices:\n(A) variable.\n(B) determined by the base sequence in RNA.\n(C) genetically determined.\n(D) always 1:1.<｜Assistant｜>\n<｜User｜>assistant\n\n\n\n\nThe correct answer is (A) variable.<｜Assistant｜>\n'